# 00 · Discover ports and connect

Enumerate serial adapters without opening or probing them. Adapter metadata does not identify an attached chiller.

Run top to bottom in a **Python 3.12+** kernel. This notebook includes all configuration and setup; no sibling example files are imported. Run only one hardware notebook/process at a time.

**Prepare the filled cooling loop before connecting.** T255P connection sends U, takes serial control and can change AUTO START to RUN, even with target writes disabled. Software disconnect does not stop the pump/chiller or restore its target. Physical hardware has not been validated with this package.

## 1. Environment and imports

From the repository root, run `uv sync --locked --group notebooks`, then `uv run --locked --group notebooks jupyter lab`. Select the environment kernel described in examples/README.md. The script-only dashboard (06_dashboard.py) needs `--extra gui`. Restart the kernel after installation or driver source changes. Imports open no serial ports.

In [1]:
import sys
from pathlib import Path

# Prefer checkout sources when launched from the repository or examples folder.
for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "src/tark_chiller/controller.py").is_file():
        sys.path.insert(0, str(candidate / "src"))
        break

In [2]:
from serial.tools import list_ports

from tark_chiller.controller import T255P, TarkChiller

## Discover available adapters

Run this cell before configuring the device. Identify the adapter physically before copying its port into the configuration. No device is opened or probed.

In [3]:
ports = sorted(list_ports.comports(), key=lambda port: port.device)
print("Available serial ports (no devices probed):")
for port in ports:
    print(f"  {port.device}: {port.description}")
    print(f"    Hardware ID: {port.hwid}")
if not ports:
    print("  No serial ports reported by the operating system.")

Available serial ports (no devices probed):
  COM3: USB Serial Port (COM3)
    Hardware ID: USB VID:PID=0403:6001 SER=VDIS200A
  COM6: Windfreak Serial USB Devices (COM6)
    Hardware ID: USB VID:PID=0483:A3E5 SER=209233635232 LOCATION=1-7.3.3.1.4
  COM7: USB Serial Port (COM7)
    Hardware ID: USB VID:PID=0403:6015 SER=D200MMC5A
  COM9: USB Serial Port (COM9)
    Hardware ID: USB VID:PID=0403:6001 SER=FTB8W4Y7A


## 2. Choose the port

This example connects to a **ThermoTek T255P**. Set `PORT` to its serial port (for example `"COM9"`). Adapter metadata identifies the USB adapter, not the attached chiller.

The connection cell below spells out the T255P profile: 9600 baud, 8N1, XON/XOFF, no hardware flow control, and RTS/DTR deasserted. Check that your adapter/cable supports these line levels. Distilled-water target bounds are 5–45 °C, in 0.1 °C steps. See the [hardware guide](../docs/hardware.md) for wiring and the separate CAL protocol.

In [4]:
PORT = "COM9"  # Your selected T255P serial adapter.

## 3. Connection settings

The next operation cell constructs the driver directly with the settings shown. Construction opens nothing. Its context connects and releases the port, including on a read error or interrupt. Among the notebooks, only example 02 enables target writes; dashboard script 06 also enables explicit GUI target requests.

## 4. Discover ports and connect

Enumerate serial adapters without opening or probing them. Adapter metadata does not identify an attached chiller.

Run the next cell as a unit: it owns connection and cleanup together. Interrupt the cell to cancel; errors remain visible.

In [5]:
if not isinstance(PORT, str) or not PORT.strip():
    raise ValueError('Hardware not configured: set PORT, e.g. "COM9". No port was opened.')

chiller = TarkChiller(
    {
        "port": PORT,
        "baudrate": 9600,
        "bytesize": 8,
        "parity": "N",
        "stopbits": 1,
        "xonxoff": True,
        "rtscts": False,
        "dsrdtr": False,
        "rts": False,
        "dtr": False,
    },
    protocol=T255P(allow_writes=False),
    setpoint_range=(5.0, 45.0),
    coolant_source="ThermoTek T255P Rev C pp9/20; verify installed distilled water",
)

print(f"Connecting to configured port: {chiller.settings['port']}")
with chiller:
    print(f"Connected: {chiller.is_connected}")
print(f"Connected after cleanup: {chiller.is_connected}")

Connecting to configured port: COM9
Connected: True
Connected after cleanup: False


## 5. Check results and finish

Successful connection establishes the protocol handshake, not thermal performance.

Context cleanup closes software resources, including after a read error or kernel interrupt. If cleanup fails, resources may remain owned: allow pending I/O to finish and retry `chiller.disconnect(timeout_s=5.0)` if `chiller` was created. Do not rerun configuration over a live connection. A forced kernel termination cannot guarantee cleanup. Follow the equipment's physical shutdown procedure separately.